In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
os.chdir('C:\\Users\\Julian\\Desktop\\Cursos\\sets\\buses')
os.listdir()

df = pd.read_excel('VARIABLES LIMPIAS.xlsx')
df.head(2)

,ID,Model,Años,Tipologia,Capacidad,Marca,CILINDRAJE,Combustible,Fecha de Varada,Hora Varada,...,Fecha de revisión,Descripción de la revisión,Revisor,Fecha de habilitación,Habilitador,Zona,Ruta,Concatenar,Revisión,Tiempo habilitación
0,934,2021,4,BUS,80,VOLVO,7146.0,DIESEL,2024-01-01,05:55:52,...,2024-01-01 05:56:09,Móvil 407185/ O.T. 587154/ SE REALIZA CAMBIO ...,FLOTA-Isabel Cristina Ramirez Sanchez,2024-01-02 03:43:33,FLOTA-Isabel Cristina Ramirez Sanchez,Calle 80,669,Calle 80669,En Operación,21.794722
1,6339,2015,10,BUS,50,CHEVROLET,4570.0,DIESEL,2024-01-01,05:56:37,...,2024-01-01 05:56:52,Móvil 104397/ O.T. 751216/ REVISIÓN DE MÓVIL ...,FLOTA- MARCELA HERRERA,2024-01-01 09:57:43,FLOTA- MARCELA HERRERA,Usaquen,BK904,UsaquenBK904,En Operación,4.018333


In [3]:
df.shape, df.columns

((116215, 24),
 Index(['ID', 'Model', 'Años', 'Tipologia', 'Capacidad', 'Marca', 'CILINDRAJE',
        'Combustible', 'Fecha de Varada', 'Hora Varada', 'Franja',
        'Causa de la inmovilización', 'Descripción de la novedad', 'Creador',
        'Fecha de revisión', 'Descripción de la revisión', 'Revisor',
        'Fecha de habilitación', 'Habilitador', 'Zona', 'Ruta', 'Concatenar',
        'Revisión', 'Tiempo habilitación'],
       dtype='object'))

In [4]:
df['Causa de la inmovilización'].unique()

array(['CARROCERÍA EXTERNA', 'ACELERACIÓN - POTENC', 'VANDALISMO',
       'SUSPENSIÓN', 'COMPRESOR', 'PUERTAS', 'SILLAS - PASAMANOS',
       'SE APAGA Y NO ENCIEN', 'FUGA DE AIRE', 'EMBRAGUE', 'CORREAS',
       'CAJA - TRANSMISIÓN', 'ACCIDENTE DE TRÁNSIT', 'FRENOS',
       'SISTEMA ELÉCTRICO', 'TESTIGOS-TABL INSTRU', 'RUTERO - INFORMADOR',
       'TEMPERATURA', 'DIRECCIÓN', 'COMBUSTIBLE', 'CARROCERÍA INTERNA',
       'FUGA DE ACEITE', 'FUGA DE REFRIGERANTE', 'RODAMIENTO LLANTA',
       'SEGURIDAD', 'MOTOR', 'TURBO', 'ACCESIBILIDAD DISCAP', 'CARDAN',
       'PERNO FIJACIÓN LLANT', 'AMBIENTAL', 'DOCUMENTOS', 'TROQUE ',
       'SISTEMA NEUMÁTICO', 'LIMPIAPARABRISAS', 'INMOV POR TRANSITO',
       'LLANTA SUELTA', 'SISTEMA ELÉCTRICO ', 'PUERTAS ',
       'CARROCERÍA EXTERNA ', 'DIRECCIÓN ', 'FUGA DE ACEITE ',
       'COMBUSTIBLE ', 'LLANTA LISA', 'SIRCI CONTROL',
       'CAJA - TRANSMISIÓN ', 'LIMPIAPARABRISAS ', 'MOTOR ',
       'CARROCERÍA INTERNA ', 'ZZZZZZZZZZ-O. CAU', 'ESTRUC - CHASIS 

### Causas que potencialmente podrían ser explicadas 

Pregunta ha una IA que hace las veces del experto técnico

Estas están más relacionadas con el desgaste mecánico, configuración del vehículo o condiciones operativas:

1. FRENOS, EMBRAGUE, CAJA - TRANSMISIÓN, MOTOR, TURBO, CORREAS, SUSPENSIÓN, SISTEMA NEUMÁTICO, SISTEMA ELÉCTRICO

Explicables por: Años, CILINDRAJE, Marca, Modelo, Combustible, Tipologia, Capacidad, Fecha/Hora de varada.

2. FUGA DE AIRE, FUGA DE ACEITE, FUGA DE REFRIGERANTE, TEMPERATURA

Posiblemente correlacionadas con el desgaste (edad del bus), o características mecánicas (CILINDRAJE, Marca).

3. LLANTA LISA, RODAMIENTO LLANTA, PERNO FIJACIÓN LLANT

Explicables si se conoce frecuencia de uso por ruta, edad del vehículo, Zona, Capacidad.

4. PUERTAS, SILLAS - PASAMANOS, CARROCERÍA INTERNA/EXTERNA

Relacionadas con el desgaste físico y el trato de los pasajeros (posible proxy: Zona, Ruta, Tipología, Capacidad).

5. ACELERACIÓN - POTENCIA, DIRECCIÓN, CARDAN

Variables como CILINDRAJE, Modelo, Marca podrían ser explicativas.

In [5]:
# Definamos nuestro vector de salida y

causas_explicables = [
    'FRENOS', 'EMBRAGUE', 'CAJA - TRANSMISIÓN', 'MOTOR',
    'TURBO', 'CORREAS', 'SUSPENSIÓN', 'SISTEMA NEUMÁTICO',
    'SISTEMA ELÉCTRICO'
]


df['Causa de la inmovilización'] = df['Causa de la inmovilización'].str.strip().str.upper()
y = df['Causa de la inmovilización'].isin(causas_explicables).astype(int)

In [6]:
y.value_counts()

Causa de la inmovilización
0    82807
1    33408
Name: count, dtype: int64

In [7]:
# Definamos nuestra matriz de caracteristicas X

variables_X = ['Model','Años','Tipologia','Capacidad','Marca','CILINDRAJE','Combustible','Franja', 'Zona']


# Selección de variables
X = df[variables_X].copy()

X.head()

,Model,Años,Tipologia,Capacidad,Marca,CILINDRAJE,Combustible,Franja,Zona
0,2021,4,BUS,80,VOLVO,7146.0,DIESEL,a. m.,Calle 80
1,2015,10,BUS,50,CHEVROLET,4570.0,DIESEL,a. m.,Usaquen
2,2014,11,BUS,50,MERCEDES BENZ,4249.0,DIESEL,a. m.,San Cristobal
3,2021,4,BUS,80,SCANIA,9290.0,GNV,a. m.,Ciudad Bolivar
4,2020,5,BUS,80,SCANIA,9290.0,GNV,a. m.,Kennedy


In [8]:
# Codificación de variables

X.Franja.value_counts()

Franja
p. m.    45534
a. m.    34904
Name: count, dtype: int64

In [9]:
X = pd.get_dummies(X, columns=['Franja'], prefix='Franja', drop_first=True)
X.head()

,Model,Años,Tipologia,Capacidad,Marca,CILINDRAJE,Combustible,Zona,Franja_p. m.
0,2021,4,BUS,80,VOLVO,7146.0,DIESEL,Calle 80,False
1,2015,10,BUS,50,CHEVROLET,4570.0,DIESEL,Usaquen,False
2,2014,11,BUS,50,MERCEDES BENZ,4249.0,DIESEL,San Cristobal,False
3,2021,4,BUS,80,SCANIA,9290.0,GNV,Ciudad Bolivar,False
4,2020,5,BUS,80,SCANIA,9290.0,GNV,Kennedy,False


In [10]:
X = pd.get_dummies(X, columns= ['Tipologia'], prefix= 'Tipologia', drop_first= True)
X.head(3)

,Model,Años,Capacidad,Marca,CILINDRAJE,Combustible,Zona,Franja_p. m.,Tipologia_BUS
0,2021,4,80,VOLVO,7146.0,DIESEL,Calle 80,False,True
1,2015,10,50,CHEVROLET,4570.0,DIESEL,Usaquen,False,True
2,2014,11,50,MERCEDES BENZ,4249.0,DIESEL,San Cristobal,False,True


In [11]:
X.Marca.nunique(), X.Combustible.nunique(), X.Zona.nunique()

(12, 5, 18)

In [12]:
print(X.Combustible.unique())

X['Combustible'] = X['Combustible'].replace({
    'ELÉCTRICO': 'ELECTRICO',
    'EL�CTRICO': 'ELECTRICO',
    'HIBRIDO (DIESEL - ELECTRICO)': 'HIBRIDO',
})

X['Combustible'].unique()

['DIESEL' 'GNV' 'ELECTRICO' 'HIBRIDO (DIESEL - ELECTRICO)' 'EL�CTRICO']


array(['DIESEL', 'GNV', 'ELECTRICO', 'HIBRIDO'], dtype=object)

In [13]:
X.Marca.unique()

array(['VOLVO', 'CHEVROLET', 'MERCEDES BENZ', 'SCANIA', 'BYD', 'HINO',
       'AGRALE', 'THOMAS BUILT', 'VOLKSWAGEN', 'BLUEBIRD',
       'INTERNACIONAL', 'YUTONG'], dtype=object)

In [14]:
X.Zona.unique()

array(['Calle 80', 'Usaquen', 'San Cristobal', 'Ciudad Bolivar',
       'Kennedy', 'Engativa', 'Fontibon IV', 'Perdomo II', 'Usme III',
       'Bosa', 'Usme II', 'Suba Centro III', 'Suba Oriental',
       'Tintal - Zona Franca', 'Fontibon V', 'Fontibon III',
       'Fontibon II', 'Suba Centro VI'], dtype=object)

In [15]:
# Aplicamos one-hot encoding 

X = pd.get_dummies(X, columns=['Marca'], prefix='Marca', drop_first=True)
X.head(2)

,Model,Años,Capacidad,CILINDRAJE,Combustible,Zona,Franja_p. m.,Tipologia_BUS,Marca_BLUEBIRD,Marca_BYD,Marca_CHEVROLET,Marca_HINO,Marca_INTERNACIONAL,Marca_MERCEDES BENZ,Marca_SCANIA,Marca_THOMAS BUILT,Marca_VOLKSWAGEN,Marca_VOLVO,Marca_YUTONG
0,2021,4,80,7146.0,DIESEL,Calle 80,False,True,False,False,False,False,False,False,False,False,False,True,False
1,2015,10,50,4570.0,DIESEL,Usaquen,False,True,False,False,True,False,False,False,False,False,False,False,False


In [16]:
X = pd.get_dummies(X, columns=['Zona'], prefix='Zona', drop_first=True)
X.head(2)

,Model,Años,Capacidad,CILINDRAJE,Combustible,Franja_p. m.,Tipologia_BUS,Marca_BLUEBIRD,Marca_BYD,Marca_CHEVROLET,...,Zona_Kennedy,Zona_Perdomo II,Zona_San Cristobal,Zona_Suba Centro III,Zona_Suba Centro VI,Zona_Suba Oriental,Zona_Tintal - Zona Franca,Zona_Usaquen,Zona_Usme II,Zona_Usme III
0,2021,4,80,7146.0,DIESEL,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2015,10,50,4570.0,DIESEL,False,True,False,False,True,...,False,False,False,False,False,False,False,True,False,False


In [17]:
X = pd.get_dummies(X, columns=['Combustible'], prefix='Combustible', drop_first=True)
X.head(2)

,Model,Años,Capacidad,CILINDRAJE,Franja_p. m.,Tipologia_BUS,Marca_BLUEBIRD,Marca_BYD,Marca_CHEVROLET,Marca_HINO,...,Zona_Suba Centro III,Zona_Suba Centro VI,Zona_Suba Oriental,Zona_Tintal - Zona Franca,Zona_Usaquen,Zona_Usme II,Zona_Usme III,Combustible_ELECTRICO,Combustible_GNV,Combustible_HIBRIDO
0,2021,4,80,7146.0,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2015,10,50,4570.0,False,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False


In [18]:
X.drop('Model', axis=1, inplace= True)
X.head(2)

,Años,Capacidad,CILINDRAJE,Franja_p. m.,Tipologia_BUS,Marca_BLUEBIRD,Marca_BYD,Marca_CHEVROLET,Marca_HINO,Marca_INTERNACIONAL,...,Zona_Suba Centro III,Zona_Suba Centro VI,Zona_Suba Oriental,Zona_Tintal - Zona Franca,Zona_Usaquen,Zona_Usme II,Zona_Usme III,Combustible_ELECTRICO,Combustible_GNV,Combustible_HIBRIDO
0,4,80,7146.0,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,10,50,4570.0,False,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,False


In [19]:
X.isna().sum()

Años                            0
Capacidad                       0
CILINDRAJE                   6760
Franja_p. m.                    0
Tipologia_BUS                   0
Marca_BLUEBIRD                  0
Marca_BYD                       0
Marca_CHEVROLET                 0
Marca_HINO                      0
Marca_INTERNACIONAL             0
Marca_MERCEDES BENZ             0
Marca_SCANIA                    0
Marca_THOMAS BUILT              0
Marca_VOLKSWAGEN                0
Marca_VOLVO                     0
Marca_YUTONG                    0
Zona_Calle 80                   0
Zona_Ciudad Bolivar             0
Zona_Engativa                   0
Zona_Fontibon II                0
Zona_Fontibon III               0
Zona_Fontibon IV                0
Zona_Fontibon V                 0
Zona_Kennedy                    0
Zona_Perdomo II                 0
Zona_San Cristobal              0
Zona_Suba Centro III            0
Zona_Suba Centro VI             0
Zona_Suba Oriental              0
Zona_Tintal - 

In [20]:
X.shape, y.shape

((116215, 36), (116215,))

In [21]:
# Crear máscara booleana para filas sin NaN en X
mask = ~X.isna().any(axis=1)

# Filtrar X e y usando esa máscara para mantener filas iguales
X = X[mask]
y = y[mask]

In [22]:
X.shape, y.shape

((109455, 36), (109455,))

## Entrenamiento de nuestro modelo: probemos con SVM

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalado de caracteristicas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Especificación del modelo
svm = SVC(kernel='linear', random_state=42)

# Entrenamiento del modelo
svm.fit(X_train_scaled, y_train)

In [ ]:
# Predicción
y_pred = svm.predict(X_test_scaled)

# Rendimiento del modelo 
print(confusion_matrix(y_test, y_pred))


print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))